In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os


In [2]:
sys.path.append(os.path.abspath(".."))

In [3]:
from sklearn.model_selection import train_test_split
from src.data_utils import (load_data)
from src.feature_utils import (
basic_clean,
fit_missing_values,
apply_missing_values,
fit_geo_imputer,
apply_geo_imputer,
fit_log_transform,
apply_log_transform,
fit_value_replacement,
apply_value_replacement,
fit_outlier_caps,
apply_outlier_caps,
fit_rare_categories,
apply_rare_categories,
create_regular_features,
create_binary_features


)

In [4]:
X_train = load_data("../data/X_train.csv")
y_train = load_data("../data/y_train.csv")

df_raw = X_train.merge(y_train, on="id")

DataFrame successfully loaded.
Shape: (59400, 40)
DataFrame successfully loaded.
Shape: (59400, 2)


In [5]:
df = basic_clean(df_raw)

Dropped columns: ['id', 'wpt_name', 'num_private', 'subvillage', 'recorded_by', 'scheme_name', 'extraction_type_group', 'extraction_type_class', 'payment_type', 'water_quality', 'quantity_group', 'source', 'waterpoint_type_group']
 Placeholder values replaced:
{'funder': 781, 'installer': 781, 'management': 561, 'management_group': 561, 'payment': 8157, 'quality_group': 1876, 'quantity': 789, 'source_class': 278}
Converted to category: ['region_code', 'district_code']
Converted to datetime: ['date_recorded']
Dropped 535 duplicate rows


In [6]:
X = df.drop(columns=["status_group"])
y = df["status_group"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
missing_stats = fit_missing_values(X_train)

X_train = apply_missing_values(X_train, missing_stats)
X_test = apply_missing_values(X_test, missing_stats)

In [8]:
rare_stats = fit_rare_categories(X_train, cols=[
    "extraction_type",
    "waterpoint_type",
    "installer",
    "funder",
    "scheme_management",
    "management",
    "water_quality",
    "quality_group",
    "quantity",
    "payment",
    "payment_type",
    "source",
    "source_type",
    "source_class"])
X_train = apply_rare_categories(X_train, rare_stats)
X_test = apply_rare_categories(X_test, rare_stats)


In [9]:
X_train["payment"].describe(include="all").T

count         47092
unique            6
top       never pay
freq          26488
Name: payment, dtype: object

In [10]:
stats = fit_value_replacement(
    X_train,
    cols=["gps_height", "population"],
    treat_zero_as_nan=["gps_height", "population"],
    treat_negative_as_nan=["gps_height"],
    group_cols=["ward", "district_code"]
)

X_train = apply_value_replacement(
    X_train,
    stats,
    cols=["gps_height", "population"],
    treat_zero_as_nan=["gps_height", "population"],
    treat_negative_as_nan=["gps_height"],
    group_cols=["ward", "district_code"]
)

X_test = apply_value_replacement(
    X_test,
    stats,
    cols=["gps_height", "population"],
    treat_zero_as_nan=["gps_height", "population"],
    treat_negative_as_nan=["gps_height"],
    group_cols=["ward", "district_code"]
)

In [11]:
num_cols = df.select_dtypes(include=["number"]).columns

log_cols = fit_log_transform(
    X_train,
    cols=num_cols,  # optional
    skew_threshold=1.0
)

print("Skewed cols are: ", log_cols)

X_train = apply_log_transform(X_train, log_cols)
X_test  = apply_log_transform(X_test, log_cols)

Skewed cols are:  ['amount_tsh', 'population']


In [12]:
geo_stats = fit_geo_imputer(
    X_train,
    lat_col="latitude",
    lon_col="longitude"
)

X_train = apply_geo_imputer(
    X_train,
    geo_stats,
    lat_col="latitude",
    lon_col="longitude"
)

X_test = apply_geo_imputer(
    X_test,
    geo_stats,
    lat_col="latitude",
    lon_col="longitude"
)

In [13]:
X_train.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
amount_tsh,47092.0,NaN,NaN,NaN,323.616344,0.0,0.0,0.0,25.0,350000.0,3198.280571
date_recorded,47092,NaN,NaN,NaN,2012-03-27 23:42:26.878450,2004-01-07 00:00:00,2011-03-31 00:00:00,2012-10-09 00:00:00,2013-02-09 00:00:00,2013-12-03 00:00:00,NaN
funder,47092,11,others,25751,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gps_height,47092.0,NaN,NaN,NaN,1067.663998,1.0,677.75,1206.0,1409.0,2770.0,520.76631
installer,47092,11,others,22842,NaN,NaN,NaN,NaN,NaN,NaN,NaN
longitude,47092.0,NaN,NaN,NaN,35.13595,29.615157,33.341961,34.988789,37.195054,40.311285,2.583781
latitude,47092.0,NaN,NaN,NaN,-5.760562,-11.511275,-8.569185,-5.073427,-3.345998,-0.0,2.902583
basin,47092,9,Lake Victoria,7782,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,47092,21,Iringa,4218,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region_code,47092.0,27.0,11.0,4224.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
num_cols = X_train.select_dtypes(include=np.number).columns

for col in num_cols:
    print(col, "zeros:", (X_train[col] == 0).sum(), "neg:", (X_train[col] < 0).sum())

amount_tsh zeros: 32939 neg: 0
gps_height zeros: 0 neg: 0
longitude zeros: 0 neg: 0
latitude zeros: 0 neg: 47092
population zeros: 0 neg: 0
construction_year zeros: 0 neg: 0
amount_tsh_log zeros: 32939 neg: 0
population_log zeros: 0 neg: 0


In [15]:
X_train = create_regular_features(X_train)
X_train = create_binary_features(X_train)

In [16]:
X_train.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
amount_tsh,47092.0,NaN,NaN,NaN,323.616344,0.0,0.0,0.0,25.0,350000.0,3198.280571
date_recorded,47092,NaN,NaN,NaN,2012-03-27 23:42:26.878450,2004-01-07 00:00:00,2011-03-31 00:00:00,2012-10-09 00:00:00,2013-02-09 00:00:00,2013-12-03 00:00:00,NaN
funder,47092,11,others,25751,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gps_height,47092.0,NaN,NaN,NaN,1067.663998,1.0,677.75,1206.0,1409.0,2770.0,520.76631
installer,47092,11,others,22842,NaN,NaN,NaN,NaN,NaN,NaN,NaN
longitude,47092.0,NaN,NaN,NaN,35.13595,29.615157,33.341961,34.988789,37.195054,40.311285,2.583781
latitude,47092.0,NaN,NaN,NaN,-5.760562,-11.511275,-8.569185,-5.073427,-3.345998,-0.0,2.902583
basin,47092,9,Lake Victoria,7782,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,47092,21,Iringa,4218,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region_code,47092.0,27.0,11.0,4224.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
